# Model Evaluation

Compare to gage data

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
# setting up logging first or else it gets preempted by another package
import watershed_workflow
watershed_workflow.setupLogging(1)

In [ ]:
import os,sys
import logging
import numpy as np
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

import pickle
import shapely
import pandas as pd
import geopandas as gpd
pd.options.display.max_columns = None
import copy
import cftime, datetime

import watershed_workflow 
import watershed_workflow.utils
import watershed_workflow.sources
import watershed_workflow.mesh
import watershed_workflow.plot
import watershed_workflow.sources.standard_names

# set the default figure size for notebooks
plt.rcParams["figure.figsize"] = (8, 6)

In [ ]:
# Force Watershed Workflow to pull data from this directory rather than a shared data directory.
# This picks up the Coweeta-specific datasets set up here to avoid large file downloads for 
# demonstration purposes.
#
def splitPathFull(path):
    """
    Splits an absolute path into a list of components such that
    os.path.join(*splitPathFull(path)) == path
    """
    parts = []
    while True:
        head, tail = os.path.split(path)
        if head == path:  # root on Unix or drive letter with backslash on Windows (e.g., C:\)
            parts.insert(0, head)
            break
        elif tail == path:  # just a single file or directory
            parts.insert(0, tail)
            break
        else:
            parts.insert(0, tail)
            path = head
    return parts

cwd = splitPathFull(os.getcwd())
assert cwd[-1] == 'workflow'
cwd = cwd[:-1]

# Note, this directory is where downloaded data will be put as well
data_dir = os.path.join(*(cwd + ['input_data',]))
def toInput(filename):
    return os.path.join(data_dir, filename)

output_dir = os.path.join(*(cwd + ['output_data',]))
output_filenames = dict()
def fromOutput(filename):
    return os.path.join(output_dir, filename)    

def toOutput(role, filename):
    output_filenames[role] = filename
    return fromOutput(filename)

# check output and input dirs exist
if not os.path.isdir(data_dir):
    os.makedirs(data_dir, exist_ok=True)
if not os.path.isdir(output_dir):
    os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Set the data directory to the local space to get the locally downloaded files
# REMOVE THIS CELL for general use outside fo Coweeta
watershed_workflow.utils.setDataDirectory(data_dir)

In [ ]:
## Parameters cell -- this provides all parameters that can be changed via pipelining to generate a new watershed. 
name = 'RussianRiver'
hucs = ['18010110'] # a list of HUCs to run


# -- parameters to clean and reduce the river network prior to meshing
prune_by_area = 20               # km^2
simplify = 200                   # length scale to target average edge 

# -- mesh triangle refinement control
refine_d0 = 200
refine_d1 = 600

refine_L0 = 200
refine_L1 = 500

refine_A0 = refine_L0**2 / 2
refine_A1 = refine_L1**2 / 2


# Refine triangles if they get too acute
min_angle = 20 # degrees

# width of reach by stream order (order:width)
river_widths = dict({1:10, 2:10, 3:20, 4:30, 5:30}) 


# Note that, by default, we tend to work in the DayMet CRS because this allows us to avoid
# reprojecting meteorological forcing datasets.
crs = watershed_workflow.crs.default_crs


# start and stop time for simulation
# note that this is the overlap of AORC and MODIS
start = cftime.DatetimeGregorian(2007, 8, 1)
end = cftime.DatetimeGregorian(2020, 12, 31)

start_noleap = cftime.DatetimeNoLeap(2007, 8, 1)
end_noleap = cftime.DatetimeNoLeap(2020, 12, 31)
cyclic_nyears = 10


## Reload data

In [ ]:
gages = gpd.read_parquet(fromOutput('03d_gages_found.parquet'))    
gages = gages[gages['agency_cd'] == 'USGS']
gages['name'] = [f'USGS-{siteno}' for siteno in gages.site_no]
gages.set_index('name', inplace=True)
gages

In [ ]:
eval_start, eval_end = pd.Timestamp('2018-10-1'), pd.Timestamp('2020-9-30')


In [ ]:
# raw data
streamflow = pd.read_csv(fromOutput('01_discharge_observations.csv'))

# streamflow times are all UTC times, but correspond to 0:00 local time (PDT or PST) on the given date
#
# as a result, the direct reading via pd.to_datetime() computes a time series that is either 8:00 UTC or 7:00 UTC depending upon Daylight Savings Time
#
# rather than mess with this, we convert to a datetime manually, setting it to 0:00 on the date.  Note this is easy because we are using DAILY data

def toDatetime(time_str):
    date = time_str.split()[0]
    return pd.to_datetime(date)

# not sure why this column name is messed up -- fix this in workflow 01
streamflow['time [date]'] = streamflow.pop('Unnamed: 0').apply(toDatetime)
streamflow.set_index('time [date]', inplace=True)
streamflow = streamflow.loc[eval_start:eval_end]

names = sorted(gages.index)
streamflow = streamflow[names]
print(len(streamflow))


In [ ]:
print(streamflow.index[0], streamflow.index[-1])

## Load Simulation data

In [ ]:
print(start)
start_as_datetime = datetime.datetime(start.year, start.month, start.day)



In [ ]:
# load and format water balance data 
ats_wb = pd.read_csv('../02_RussianRiver_transient/run-comb/water_balance_computational_domain.csv', comment='#')
print('Initial time = ', ats_wb['time [d]'][0])

ats_wb_time_delta = np.array([datetime.timedelta(days=dt) for dt in ats_wb.pop('time [d]')])
ats_wb['time [date]'] = start_as_datetime + ats_wb_time_delta


In [ ]:
# NOTE: there is a time shift interpretation gap here.

# The gage data comes in as a date that refers to the average discharge over the course of that date.

# The ATS simulation uses time_integrated=True to track the time-integral over the previous observation period.  
# But it observes things at the current time.  So the time corresponds to the time at the end of the interval -- 
# the actual total discharge (mol / day) over the PREVIOUS day.  Since the timestamp is 0:00, the interval actually 
# corresponds to the PREVIOUS day.

# So we first subtract off a day... now the DATE is correct, and we are ignoring the time anyway.
ats_wb['time [date]'] = ats_wb['time [date]'] - datetime.timedelta(days=1)
ats_wb.set_index('time [date]', drop=False, inplace=True)
print('Simulation Initial time = ', ats_wb.index[0])
print('Simulation Final time = ', ats_wb.index[-1])

# now drop all but the time window we need
ats_wb = ats_wb.loc[eval_start:eval_end]


In [ ]:
# load and format streamflow data
ats_streamflow = pd.read_csv('../02_RussianRiver_transient/run-comb/gage_discharge.csv', comment='#')

ats_streamflow_time_delta = np.array([datetime.timedelta(days=dt) for dt in ats_streamflow.pop('time [d]')])
ats_streamflow['time [date]'] = start_as_datetime + ats_streamflow_time_delta

ats_streamflow['time [date]'] = ats_streamflow['time [date]'] - datetime.timedelta(days=1)
ats_streamflow.set_index('time [date]', inplace=True)
ats_streamflow = ats_streamflow.loc[eval_start:eval_end]


# convert mol / s --> m^3 / s
for k in list(ats_streamflow.keys()):
    kout = k.split()[0]
    ats_streamflow[kout] = ats_streamflow.pop(k) / 55000. / 86400.

ats_streamflow = ats_streamflow[sorted(ats_streamflow.keys())]

In [ ]:
# align the times:
# streamflow comes from NWIS, and it is daily data, but the timestamp is 7:00 UTC, or 0:00 local time.
# streamflow_ats is simulated based on a time-integrated quantity that is relative to a timestamp 

print(streamflow.index[0], streamflow.index[-1])
print(ats_streamflow.index[0], ats_streamflow.index[-1])
print(ats_wb.index[0], ats_wb.index[-1])

print(len(streamflow), len(ats_streamflow), len(ats_wb))

## Plot comparisons of discharge

In [ ]:
# first compare precip to gage evaluation data -- did we get the right years/forcing lineup?
fig, ax = plt.subplots(1,1)
ax.plot(ats_wb['time [date]'], ats_wb['rain precipitation [m d^-1]'], 'b')
ax2 = ax.twinx()

for k in streamflow.keys():
    ax2.plot(streamflow.index, streamflow[k], 'k')

ax.set_xlabel('time [date]')
ax.set_ylabel('precip [m s^-1]')
ax2.set_ylabel('discharge [mol s^-1]')
ax.set_xlim(pd.Timestamp('2018-10-1'), pd.Timestamp('2020-10-1'))


In [ ]:
# next compare gage to ATS gage evaluation data

def plot(ax, k):
    k_ats = k
    k_obs = k
    #assert(k_ats.split()[0] == k_obs.split()[-1])
    ax2 = ax.twinx()
    ax2.invert_yaxis()  
    ax2.plot(ats_wb['time [date]'], ats_wb['rain precipitation [m d^-1]'], 'b', linewidth=0.7)

    ax.plot(streamflow.index, streamflow[k_obs], 'k', linewidth=0.9)
    ax.plot(ats_streamflow.index, ats_streamflow[k_ats], 'goldenrod', linewidth=0.9)

    ax.set_xlabel('time [date]')
    ax2.set_ylabel('precip [m s^-1]')
    ax.set_ylabel('discharge [mol s^-1]')
    ax.set_title(k_ats)

def plotOne(k):
    fig, axs = plt.subplots(1,1)
    plot(axs, k)
    plt.tight_layout()
    plt.show()      
         
plotOne('USGS-11463682')



In [ ]:
plotOne('USGS-11465240')

In [ ]:
# next compare gage to ATS gage evaluation data
fig, axs = plt.subplots(9,3, figsize=(10,20))
axs = axs.ravel()

for ax, k_ats, k_obs in zip(axs, ats_streamflow.keys(), streamflow.keys()):
    assert(k_ats.split()[0] == k_obs.split()[-1])
    ax2 = ax.twinx()
    ax2.invert_yaxis()  
    ax2.plot(ats_wb['time [date]'], ats_wb['rain precipitation [m d^-1]'], 'b', linewidth=0.7)

    ax.plot(streamflow.index, streamflow[k_obs], 'k', linewidth=0.9)
    ax.plot(ats_streamflow.index, ats_streamflow[k_ats], 'goldenrod', linewidth=0.9)

    ax.set_xlabel('time [date]')
    ax2.set_ylabel('precip [m s^-1]')
    ax.set_ylabel('discharge [mol s^-1]')
    ax.set_title(k_ats)

plt.tight_layout()
plt.show()

In [ ]:
import hydroeval
hydroeval

In [ ]:
gages['KGE'] = [hydroeval.evaluator(hydroeval.kge, ats_streamflow[k], streamflow[k]) for k in gages.index]

In [ ]:
def computeKGE(k):
    mask = np.where(~np.isnan(streamflow[k].values))
    print(len(streamflow[k]), len(mask[0]))
    streamflow_k = streamflow[k].values[mask]
    ats_streamflow_k = ats_streamflow[k].values[mask]
    return hydroeval.evaluator(hydroeval.kge, ats_streamflow_k, streamflow_k)

kge = np.array([computeKGE(k) for k in gages.index])

In [ ]:
gages['KGE'] = kge[:,0,0]
gages['KGE_r'] = kge[:,1,0]
gages['KGE_alpha'] = kge[:,2,0]
gages['KGE_beta'] = kge[:,3,0]
gages['nKGE'] = (gages['KGE'] + np.sqrt(2) - 1) / np.sqrt(2)

In [ ]:
gages[['nKGE', 'KGE_r', 'KGE_alpha', 'KGE_beta']]


In [ ]:
any(np.isnan(streamflow['USGS-11462000'].values))

In [ ]:
np.nanmedian(gages['nKGE'].values)

In [ ]:
print(np.nanmedian(gages['KGE_r']))
print(np.nanmedian(gages['KGE_alpha']))
print(np.nanmedian(gages['KGE_beta']))
